In [1]:
import pandas as pd
import numpy as np
import statistics
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
gpu = gpus[0]

tf.config.experimental.set_memory_growth(gpu, True)
import transformers
from sklearn.metrics import confusion_matrix, classification_report

2022-09-23 22:42:23.910310: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:937] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-09-23 22:42:24.008837: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:937] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-09-23 22:42:24.009622: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:937] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero


In [2]:
transformers.logging.set_verbosity_error()

In [3]:
data=pd.read_csv('../input/circa-dataset/circa-data.tsv',sep='\t')
data.head()

,id,context,question-X,canquestion-X,answer-Y,judgements,goldstandard1,goldstandard2
0,0,Y has just travelled from a different city to ...,Are you employed?,I am employed .,I'm a veterinary technician.,Yes#Yes#Yes#Yes#Yes,Yes,Yes
1,1,X wants to know about Y's food preferences.,Are you a fan of Korean food?,I am a fan of Korean food .,I wouldn't say so,Probably no#No#No#No#Probably yes / sometimes yes,No,No
2,2,Y has just told X that he/she is thinking of b...,Are you bringing any pets into the flat?,I am bringing pets into the flat .,I do not own any pets,No#No#No#No#No,No,No
3,3,X wants to know what activities Y likes to do ...,Would you like to get some fresh air in your f...,I would like to get fresh air in my free time .,I am desperate to get out of the city.,"Yes#Yes, subject to some conditions#Probably y...",Yes,Yes
4,4,X and Y are childhood neighbours who unexpecte...,Is your family still living in the neighborhood?,My family is living in the neighborhood .,My parents are snowbirds now.,"No#In the middle, neither yes nor no#Probably ...","In the middle, neither yes nor no","In the middle, neither yes nor no"


In [4]:
print(data.shape)
data['context'].value_counts()

(34268, 8)


Y has just told X that he/she is thinking of buying a flat in New York.             3500
Y has just travelled from a different city to meet X.                               3487
X wants to know about Y's music preferences.                                        3483
Y has just told X that he/she is considering switching his/her job.                 3479
X wants to know what activities Y likes to do during weekends.                      3465
X and Y are colleagues who are leaving work on a Friday at the same time.           3452
X wants to know what sorts of books Y likes to read.                                3445
X and Y are childhood neighbours who unexpectedly run into each other at a cafe.    3391
Y has just moved into a neighbourhood and meets his/her new neighbour X.            3356
X wants to know about Y's food preferences.                                         3210
Name: context, dtype: int64

In [5]:
data['goldstandard1'].value_counts()

Yes                                              14504
No                                               10829
Yes, subject to some conditions                   2583
Probably yes / sometimes yes                      1244
Probably no                                       1160
In the middle, neither yes nor no                  638
Other                                              504
I am not sure how X will interpret Y’s answer       63
Name: goldstandard1, dtype: int64

In [6]:
data['goldstandard2'].value_counts()

Yes                                  16628
No                                   12833
Yes, subject to some conditions       2583
In the middle, neither yes nor no      949
Other                                  504
Name: goldstandard2, dtype: int64

In [7]:
#data = data.dropna()
print(data.isnull().sum())
# data['goldstandard1'].value_counts()

id                  0
context             0
question-X          0
canquestion-X      10
answer-Y            0
judgements          0
goldstandard1    2743
goldstandard2     771
dtype: int64


In [8]:
data = data.dropna()
data['goldstandard2'].value_counts()

Yes                                  15745
No                                   11985
Yes, subject to some conditions       2580
In the middle, neither yes nor no      701
Other                                  504
Name: goldstandard2, dtype: int64

In [9]:
# data['label'] = data['goldstandard1'].copy()

# data['label'] = data['label'].map({'Yes': 0, 'No': 1, 'Yes, subject to some conditions': 2,
#                                   'Probably yes / sometimes yes': 3, 'Probably no': 4,
#                                   'In the middle, neither yes nor no': 5, 'Other': 6,
#                                   'I am not sure how X will interpret Y’s answer': 7})

In [10]:
data['label_2'] = data['goldstandard2'].copy()

data['label_2'] = data['label_2'].map({'Yes': 0, 'No': 1, 'Yes, subject to some conditions': 2,
                                  'In the middle, neither yes nor no': 3, 'Other': 4})

In [11]:
data['label_2'].value_counts()

0    15745
1    11985
2     2580
3      701
4      504
Name: label_2, dtype: int64

In [12]:
data.reset_index(drop=True, inplace=True)

In [13]:
#Length of avg question:
q = []
for x in range(data.shape[0]):
    q.append(len(data['question-X'][x].split()))
    
#Length of avg answer:
a = []
for x in range(data.shape[0]):
    a.append(len(data['answer-Y'][x].split()))
    
print(statistics.mean(q))
print(statistics.mean(a))

6.602189433603046
5.699095668729177


In [14]:
from sklearn.model_selection import train_test_split

train , test = train_test_split(data, test_size = 0.20)

# train['sep_token'] = '[SEP]'
# train['cls_token'] = '[CLS]'
# train['text'] = train['cls_token'] + \
#                     train['context'] + train['sep_token']+ train['question-X'] + \
#                     train['sep_token'] + train['answer-Y'] + \
#                 train['sep_token']


X = train[['question-X', 'answer-Y']]
y = tf.keras.utils.to_categorical(train.label_2, num_classes=5)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.15)

print(train.shape, test.shape)
print(X_train.shape)
print(X_val.shape)
print(y_train.shape)
print(y_val.shape)

(25212, 9) (6303, 9)
(21430, 2)
(3782, 2)
(21430, 5)
(3782, 5)


In [15]:
max_length = None
epochs = 20
batch_size = 32

In [16]:
class BertSemanticDataGenerator(tf.keras.utils.Sequence):
    def __init__(
        self,
        sentence_pairs,
        labels,
        batch_size=batch_size,
        shuffle=True,
        include_targets=True,
    ):
        self.sentence_pairs = sentence_pairs
        self.labels = labels
        self.shuffle = shuffle
        self.batch_size = batch_size
        self.include_targets = include_targets
        # Load BERT Tokenizer to encode the text.
        # base-base-uncased pretrained model.
        self.tokenizer = transformers.BertTokenizer.from_pretrained(
            "bert-base-uncased", do_lower_case=True
        )
        self.indexes = np.arange(len(self.sentence_pairs))
        self.on_epoch_end()

    def __len__(self):
        # Denotes the number of batches per epoch.
        return len(self.sentence_pairs) // self.batch_size

    def __getitem__(self, idx):
        # Retrieves the batch of index.
        indexes = self.indexes[idx * self.batch_size : (idx + 1) * self.batch_size]
        sentence_pairs = self.sentence_pairs[indexes]

        # With BERT tokenizer's batch_encode_plus, a batch of both the sentences are
        # encoded together and separated by [SEP] token.
        encoded = self.tokenizer.batch_encode_plus(
            sentence_pairs.tolist(),
            add_special_tokens=True,
            max_length=max_length,
            return_attention_mask=True,
            return_token_type_ids=True,
            pad_to_max_length=True,
            return_tensors="tf",
        )

        input_ids = np.array(encoded["input_ids"], dtype="int32")
        attention_masks = np.array(encoded["attention_mask"], dtype="int32")
        token_type_ids = np.array(encoded["token_type_ids"], dtype="int32")

        # Set to True if data generator is used for training/validation.
        if self.include_targets:
            labels = np.array(self.labels[indexes], dtype="int32")
            return [input_ids, attention_masks, token_type_ids], labels
        else:
            return [input_ids, attention_masks, token_type_ids]

    def on_epoch_end(self):
        # Shuffle indices after each epoch, if shuffle is set to True.
        if self.shuffle:
            np.random.RandomState(42).shuffle(self.indexes)

In [17]:
input_ids = tf.keras.layers.Input(
    shape=(max_length,), dtype=tf.int32, name="input_ids"
)
attention_masks = tf.keras.layers.Input(
    shape=(max_length,), dtype=tf.int32, name="attention_masks"
)
token_type_ids = tf.keras.layers.Input(
    shape=(max_length,), dtype=tf.int32, name="token_type_ids"
)
# Loading pretrained BERT model.
bert_model = transformers.TFBertModel.from_pretrained("bert-base-uncased")
# Freeze the BERT model to reuse the pretrained features without modifying them.
bert_model.trainable = False

bert_output = bert_model(
    input_ids, attention_mask=attention_masks, token_type_ids=token_type_ids
)
sequence_output = bert_output.last_hidden_state
pooled_output = bert_output.pooler_output
#dropout = tf.keras.layers.Dropout(0.1)(pooled_output)
clf_output = sequence_output[:, 0, :]
output = tf.keras.layers.Dense(5, activation="softmax")(clf_output)
model = tf.keras.models.Model(
    inputs=[input_ids, attention_masks, token_type_ids], outputs=output
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="categorical_crossentropy",
    metrics=["acc"],
)

2022-09-23 22:42:28.204304: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2022-09-23 22:42:28.204805: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:937] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-09-23 22:42:28.205553: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:937] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-09-23 22:42:28.206229: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:937] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA 

Downloading:   0%|          | 0.00/570 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/511M [00:00<?, ?B/s]

In [18]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_ids (InputLayer)          [(None, None)]       0                                            
__________________________________________________________________________________________________
attention_masks (InputLayer)    [(None, None)]       0                                            
__________________________________________________________________________________________________
token_type_ids (InputLayer)     [(None, None)]       0                                            
__________________________________________________________________________________________________
tf_bert_model (TFBertModel)     TFBaseModelOutputWit 109482240   input_ids[0][0]                  
                                                                 attention_masks[0][0]        

In [19]:
train_data = BertSemanticDataGenerator(
    X_train[['question-X', 'answer-Y']].values.astype("str"),
    y_train,
    batch_size=batch_size,
    shuffle=True,
)
valid_data = BertSemanticDataGenerator(
    X_val[['question-X', 'answer-Y']].values.astype("str"),
    y_val,
    batch_size=batch_size,
    shuffle=False,
)

Downloading:   0%|          | 0.00/226k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

In [20]:
cp_callback = tf.keras.callbacks.ModelCheckpoint('circa_qa_bert_trial.h5',
                                                monitor='val_loss',
                                                mode='min',
                                                save_best_only=True,
                                                save_weights_only=True,
                                                verbose=1)



early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss',
                                                          mode='min',
                                                          patience=3,
                                                          verbose=1)

callbacks_list = [early_stopping, cp_callback]

In [21]:
y_test = tf.keras.utils.to_categorical(test.label_2, num_classes=5)

In [22]:
test_data = BertSemanticDataGenerator(
    test[["question-X", "answer-Y"]].values.astype("str"),
    y_test,
    batch_size=batch_size,
    shuffle=False,
)

In [23]:
train.head()

,id,context,question-X,canquestion-X,answer-Y,judgements,goldstandard1,goldstandard2,label_2
7390,8197,Y has just moved into a neighbourhood and meet...,Do you like this area of town?,I like this area of town .,I am not familiar with this area.,"In the middle, neither yes nor no#In the middl...","In the middle, neither yes nor no","In the middle, neither yes nor no",3
9279,10236,X wants to know what sorts of books Y likes to...,Have you read the Harry Potter series?,I have read the Harry Potter series .,I read those as a child.,Yes#Yes#Yes#Yes#Yes,Yes,Yes,0
23534,25617,Y has just told X that he/she is considering s...,Do you work at an office?,I work at an office .,I work outdoors.,No#No#No#No#No,No,No,1
1566,1758,Y has just moved into a neighbourhood and meet...,Are you staying long-term?,I am staying long-term .,My job contract is only a year.,I am not sure how X will interpret Y’s answer#...,Probably no,No,1
14342,15753,Y has just moved into a neighbourhood and meet...,Is this the first time you live in this state?,This is the first time I live in this state .,I am from here.,No#No#No#No#No,No,No,1


In [24]:
history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=epochs,
    workers=-1, callbacks=callbacks_list,
    verbose=1)

/opt/conda/lib/python3.7/site-packages/transformers/tokenization_utils_base.py:2307: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  FutureWarning,
2022-09-23 22:43:11.452210: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:185] None of the MLIR Optimization Passes are enabled (registered 2)


Epoch 1/20
669/669 [==============================] - 74s 93ms/step - loss: 0.8233 - acc: 0.6642 - val_loss: 0.6793 - val_acc: 0.7542

Epoch 00001: val_loss improved from inf to 0.67934, saving model to circa_qa_bert_trial.h5
Epoch 2/20
669/669 [==============================] - 60s 89ms/step - loss: 0.6980 - acc: 0.7294 - val_loss: 0.6324 - val_acc: 0.7622

Epoch 00002: val_loss improved from 0.67934 to 0.63241, saving model to circa_qa_bert_trial.h5
Epoch 3/20
669/669 [==============================] - 60s 90ms/step - loss: 0.6653 - acc: 0.7417 - val_loss: 0.6087 - val_acc: 0.7725

Epoch 00003: val_loss improved from 0.63241 to 0.60871, saving model to circa_qa_bert_trial.h5
Epoch 4/20
669/669 [==============================] - 60s 89ms/step - loss: 0.6523 - acc: 0.7445 - val_loss: 0.5904 - val_acc: 0.7789

Epoch 00004: val_loss improved from 0.60871 to 0.59043, saving model to circa_qa_bert_trial.h5
Epoch 5/20
669/669 [==============================] - 60s 89ms/step - loss: 0.6404 -

In [25]:
# Usually you only need <3 epochs, but there is no convergence happening here; given 
# how short the text is, increasing the epochs count to atleast some level of covergence. 

In [26]:
# Unfreeze the bert_model.
bert_model.trainable = True
# Recompile and fit again with smaller learning rate for fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_ids (InputLayer)          [(None, None)]       0                                            
__________________________________________________________________________________________________
attention_masks (InputLayer)    [(None, None)]       0                                            
__________________________________________________________________________________________________
token_type_ids (InputLayer)     [(None, None)]       0                                            
__________________________________________________________________________________________________
tf_bert_model (TFBertModel)     TFBaseModelOutputWit 109482240   input_ids[0][0]                  
                                                                 attention_masks[0][0]        

In [28]:
history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=3,
    workers=-1,verbose=1,callbacks=callbacks_list
)

Epoch 1/3
669/669 [==============================] - 111s 166ms/step - loss: 0.0536 - accuracy: 0.9827 - val_loss: 0.4740 - val_accuracy: 0.8949

Epoch 00001: val_loss did not improve from 0.31350
Epoch 2/3
669/669 [==============================] - 111s 166ms/step - loss: 0.0443 - accuracy: 0.9861 - val_loss: 0.4869 - val_accuracy: 0.8901

Epoch 00002: val_loss did not improve from 0.31350
Epoch 3/3
669/669 [==============================] - 111s 166ms/step - loss: 0.0312 - accuracy: 0.9896 - val_loss: 0.4987 - val_accuracy: 0.8957

Epoch 00003: val_loss did not improve from 0.31350


In [29]:
model.evaluate(test_data, verbose=1)

196/196 [==============================] - 13s 67ms/step - loss: 0.5782 - accuracy: 0.8929


[0.5781511068344116, 0.8928571343421936]

In [30]:
test_data_pred = BertSemanticDataGenerator(
    test[["question-X", "answer-Y"]].values.astype("str"),
    labels=None,
    batch_size=test.shape[0],
    include_targets=False,
    shuffle=False
)

In [31]:
type(test_data_pred)

__main__.BertSemanticDataGenerator

In [32]:
predicted = model.predict(test_data_pred)
predicted.shape

(6303, 5)

In [33]:
predicted_vector = predicted.copy()

In [34]:
predicted_vector[predicted_vector > 0.5] = 1
predicted_vector[predicted_vector <= 0.5] = 0
predicted_vector

array([[1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0.],
       [0., 1., 0., 0., 0.],
       ...,
       [0., 1., 0., 0., 0.],
       [0., 1., 0., 0., 0.],
       [1., 0., 0., 0., 0.]], dtype=float32)

In [35]:
print(classification_report(y_test, predicted_vector))

              precision    recall  f1-score   support

           0       0.91      0.91      0.91      3150
           1       0.89      0.90      0.89      2398
           2       0.87      0.90      0.88       509
           3       0.46      0.26      0.33       145
           4       0.92      0.98      0.95       101

   micro avg       0.90      0.89      0.89      6303
   macro avg       0.81      0.79      0.79      6303
weighted avg       0.89      0.89      0.89      6303
 samples avg       0.89      0.89      0.89      6303



/opt/conda/lib/python3.7/site-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [37]:
data['goldstandard2'].value_counts(), data['label_2'].value_counts()

(Yes                                  15745
 No                                   11985
 Yes, subject to some conditions       2580
 In the middle, neither yes nor no      701
 Other                                  504
 Name: goldstandard2, dtype: int64,
 0    15745
 1    11985
 2     2580
 3      701
 4      504
 Name: label_2, dtype: int64)

In [48]:
labels = list(set(data['label_2']))
labels_text = list(data['goldstandard2'].value_counts().index)

labels, labels_text

([0, 1, 2, 3, 4],
 ['Yes',
  'No',
  'Yes, subject to some conditions',
  'In the middle, neither yes nor no',
  'Other'])

In [49]:
def check_indirect(question, answer):
    sentence_pairs = np.array([[str(question), str(answer)]])
    test_data = BertSemanticDataGenerator(
        sentence_pairs, labels=None, batch_size=1, shuffle=False, include_targets=False,
    )

    proba = model.predict(test_data[0])[0]
    print(proba)
    idx = np.argmax(proba)
    print(idx)
    proba = f"{proba[idx]: .2f}%"
    pred = labels_text[idx]
    return pred, proba

In [50]:
question = "Do you like Italian food?"
answer = "I just had an awesome pasta yesterday for dinner"
check_indirect(question, answer)

[9.9994826e-01 1.0494828e-06 1.9082515e-06 4.8601258e-05 6.8165988e-08]
0


/opt/conda/lib/python3.7/site-packages/transformers/tokenization_utils_base.py:2307: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  FutureWarning,


('Yes', ' 1.00%')

In [51]:
question = "Do you want to catch the new James Bond movie tomorrow?"
answer = "I want to but I would prefer visiting the park more."
check_indirect(question, answer)

[1.3881367e-02 9.7939783e-01 6.5974756e-03 1.2322298e-04 1.3775367e-07]
1


('No', ' 0.98%')

In [52]:
question = "Did you enjoy the service that we provided?"
answer = "I would have to say yes to disappoinment I'm afraid"
check_indirect(question, answer)

[1.0093956e-03 8.0859876e-01 1.4892589e-04 1.9024239e-01 4.4218098e-07]
1


('No', ' 0.81%')

In [53]:
question = "Do you enjoy listening to music while coding?"
answer = "That is distracting"
check_indirect(question, answer)

[2.6647797e-07 9.9999857e-01 2.7012833e-08 1.1953618e-06 1.2537426e-09]
1


('No', ' 1.00%')

In [54]:
question = "Do you like Italian food?"
answer = "Only if it is cooked well"
check_indirect(question, answer)

[3.3411749e-07 2.1509510e-09 9.9999964e-01 1.3354419e-08 2.2970828e-11]
2


('Yes, subject to some conditions', ' 1.00%')

In [55]:
question = "Shall I pick you up from the airport?"
answer = "That would be good"
check_indirect(question, answer)

[9.9999416e-01 4.2595178e-07 3.7421412e-06 1.5891879e-06 1.6843135e-08]
0


('Yes', ' 1.00%')

In [56]:
question = "Want to go the waterpark this weekend?"
answer = "Let's get soaked!"
check_indirect(question, answer)

[9.9979872e-01 1.6449673e-04 3.3939963e-05 2.7384228e-06 8.9352568e-08]
0


('Yes', ' 1.00%')